# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azam-hussain-ml/Starter-Notebook-flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)

On Colab this clones the repo so the starter CSV is available. Locally it moves to the repo root.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/azam-hussain-ml/Starter-Notebook-flyrank-ml-internship"
REPO_DIR = "Starter-Notebook-flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

Working dir: /content/Starter-Notebook-flyrank-ml-internship


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**My lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I picked this lane for two reasons.

First, the starter data is already shaped for it. One row per page, 90-day
performance metrics, and a lifecycle column that tells me whether the page is
losing impressions. If I picked a lane that needed the warehouse from day one,
I would spend Week 1 fighting access instead of thinking about the problem.

Second, and more honestly: this is the lane where I can be wrong in a useful
way. The repo ships a hand-rule baseline that gets Precision@50 = 0.240, and a
random forest that gets 0.740 on the same slice. So there is already a number
on the board. I am not starting from "let me see what I can find" — I am
starting from "here is a score somebody already got, can I understand why, and
can I do better without cheating?" That feels like a real seven weeks of work
rather than an exploration that ends in a chart.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### The decision

A content team can realistically review about 50 pages in a week. The dataset has 30,000. So the decision is not "is this page declining" — it is **"which 50 pages does a human open first, this Monday?"**

That makes this a ranking problem, not a prediction problem. Being right about page number 900 in the list is worth nothing to anyone.

### Who acts, and what they do

A content strategist opens the page and picks one action: refresh it, expand it, protect it, prune it, or just keep monitoring. My output never touches the site. It only sets the order of the queue, and it has to give a reason for every row, otherwise the strategist has no basis to trust it or to overrule it.

### What a wrong call costs

Two directions, and they cost different things.

If I put the wrong page in the top 50, an editor spends hours refreshing a page that was fine. The wasted hours are annoying but they are not the real cost. The real cost is the page that *should* have been in that slot and instead sat untouched for another week while it kept losing impressions.

If I miss a page entirely, nobody notices, which is worse. It just keeps bleeding quietly.

Because the review budget is fixed, the metric that matters is **Precision@50**, not accuracy. A model can be 90% accurate across 30,000 rows and still put garbage in the only 50 slots anyone will ever look at.

### Why data or ML at all

A plain rule is not obviously wrong here, and I want to be careful about claiming otherwise. The repo's hand rule is transparent and a human can read it.

But the numbers say the rule leaves a lot on the table. Out of its top 50, the baseline gets about 12 right. The random forest gets about 37. That gap is the whole argument for this lane: the signal is real, but it is tangled across freshness, position, volume and engagement in a way that a fixed weighted rule does not capture well.

### The frame in one paragraph

For **a content strategist with capacity to review ~50 pages a week**, deciding **which pages to open first**, I will build **a ranked review queue with reason codes** from **the FlyRank content dataset**, scoring **each page's priority for refresh**, measured by **Precision@50**. A wrong call costs **an editor's hours on a healthy page, and another week of decline on the page they should have opened instead**. A plain rule is not enough because **the hand-rule baseline reaches only 0.240 Precision@50, and "declining" on its own is a poor trigger — as the numbers below show, over half of all pages are declining, so decline alone narrows nothing**. I will claim only **observed, directional, decision-support** results.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import pandas as pd
from pathlib import Path

CSV = Path("data/raw/content_refresh_anonymized.csv")
if not CSV.exists():                      # in case the notebook runs from repo root
    CSV = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(CSV)
declining = df["trend_direction"].eq("down")

# 1 — scale: why a human cannot triage this by hand
print(f"[1] {df.shape[0]:,} pages across {df['client_id'].nunique()} clients, "
      f"{df.shape[1]} columns")

# 2 — base rate: 'declining' is common, so it cannot be the trigger on its own
print(f"[2] declining pages: {declining.sum():,} "
      f"({declining.mean()*100:.1f}% of all pages)")

# 3 — declining but nothing at stake: the trap a naive rule walks into
low_stakes = declining & (df["impressions_90d"] < 100)
print(f"[3] declining with <100 impressions: {low_stakes.sum():,} "
      f"({low_stakes.sum()/declining.sum()*100:.1f}% of declining pages)")

# 4 — concentration: why the ORDER of the queue is the whole game
at_risk = df.loc[declining, "impressions_90d"].sort_values(ascending=False)
top50_share = at_risk.head(50).sum() / at_risk.sum() * 100
print(f"[4] the top 50 declining pages by impressions hold {top50_share:.1f}% "
      f"of all impressions among declining pages")

[1] 30,000 pages across 32 clients, 44 columns
[2] declining pages: 16,262 (54.2% of all pages)
[3] declining with <100 impressions: 3,110 (19.1% of declining pages)
[4] the top 50 declining pages by impressions hold 12.0% of all impressions among declining pages


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What my work will be able to say

- These are **observed** associations inside one 90-day snapshot of 30,000 pages and 32 clients. I watched; I did not run an experiment.
- Results are **directional**. I can say a page looks like a stronger review candidate than another page. I cannot say by how much it will recover.
- The output is **decision-support**. It orders a human's queue. It does not change any page, and every row will carry a reason code so the strategist can disagree with it.

### What my work will never be able to say

- That refreshing a page **causes** it to recover. There is no control group in this data. A page that was refreshed and then improved might have improved anyway.
- That I have reverse-engineered anything about how Google ranks pages. I only see outcomes, never the algorithm.
- That a page **will** decline next month. My label is backward-looking (see below).

### The label problem, stated up front

The starter target is `is_declining_label = (trend_direction == "down")`, and `trend_direction` is itself a threshold on `trend_pct` — down means impressions fell more than 20% versus the previous 30 days.

So this label is **defined by a rule, not observed as an outcome**. A model trained on it is partly learning to reproduce someone's −20% cutoff, not learning which pages are genuinely at risk. The lane guide calls this a proxy label and says to treat it that way, and I want to be explicit that I am treating it that way too.

For ML-02 I use it to size the problem. From Week 3, when I have the warehouse daily table, my plan is to redefine the target as a forward-window outcome — features from the prior 90 days, decline or recovery measured over the *next* 30 days. That is an observed outcome, and it is the honest version of the question I am actually asking.

### Two data traps I will not fall into

- `trend_direction` and `trend_pct` define the label, so **neither can ever be a feature.** Using them would be leakage, and the score would look wonderful and mean nothing.
- `avg_position = 0` means **"no position data"**, not rank zero (1,205 rows). A blind average over that column would quietly corrupt anything I build on position. Same logic for `fillna(0)` on the keyword columns — missingness in this dataset follows `content_type`, so filling blindly would smuggle content type into my features without me noticing.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.